### **Data Reading**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 
from pyspark.sql.window import Window

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databrcks.dfs.core.windows.net/orders')

In [0]:
display(df.limit(10))

order_id,customer_id,product_id,order_date,quantity,total_amount,year
O00001,C00710,P0159,2023-03-22T00:00:00.000Z,3,2022.87,2023
O00002,C00954,P0036,2023-06-30T00:00:00.000Z,2,3560.74,2023
O00003,C01578,P0427,2023-11-06T00:00:00.000Z,3,5903.52,2023
O00004,C00962,P0332,2024-02-27T00:00:00.000Z,3,4107.99,2024
O00005,C00156,P0038,2024-10-13T00:00:00.000Z,5,5784.95,2024
O00006,C00521,P0174,2023-05-17T00:00:00.000Z,5,407.75,2023
O00007,C00982,P0352,2024-01-18T00:00:00.000Z,4,4907.64,2024
O00008,C00976,P0172,2023-01-10T00:00:00.000Z,4,7037.88,2023
O00009,C01001,P0238,2023-04-20T00:00:00.000Z,3,4076.97,2023
O00010,C00702,P0258,2023-07-07T00:00:00.000Z,4,5695.64,2023


In [0]:
df = df.drop('_rescued_data')

order_id,customer_id,product_id,order_date,quantity,total_amount,year
O00001,C00710,P0159,2023-03-22T00:00:00.000Z,3,2022.87,2023
O00002,C00954,P0036,2023-06-30T00:00:00.000Z,2,3560.74,2023
O00003,C01578,P0427,2023-11-06T00:00:00.000Z,3,5903.52,2023
O00004,C00962,P0332,2024-02-27T00:00:00.000Z,3,4107.99,2024
O00005,C00156,P0038,2024-10-13T00:00:00.000Z,5,5784.95,2024
O00006,C00521,P0174,2023-05-17T00:00:00.000Z,5,407.75,2023
O00007,C00982,P0352,2024-01-18T00:00:00.000Z,4,4907.64,2024
O00008,C00976,P0172,2023-01-10T00:00:00.000Z,4,7037.88,2023
O00009,C01001,P0238,2023-04-20T00:00:00.000Z,3,4076.97,2023
O00010,C00702,P0258,2023-07-07T00:00:00.000Z,4,5695.64,2023


In [0]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))

order_id,customer_id,product_id,order_date,quantity,total_amount,year
O00001,C00710,P0159,2023-03-22T00:00:00.000Z,3,2022.87,2023
O00002,C00954,P0036,2023-06-30T00:00:00.000Z,2,3560.74,2023
O00003,C01578,P0427,2023-11-06T00:00:00.000Z,3,5903.52,2023
O00004,C00962,P0332,2024-02-27T00:00:00.000Z,3,4107.99,2024
O00005,C00156,P0038,2024-10-13T00:00:00.000Z,5,5784.95,2024
O00006,C00521,P0174,2023-05-17T00:00:00.000Z,5,407.75,2023
O00007,C00982,P0352,2024-01-18T00:00:00.000Z,4,4907.64,2024
O00008,C00976,P0172,2023-01-10T00:00:00.000Z,4,7037.88,2023
O00009,C01001,P0238,2023-04-20T00:00:00.000Z,3,4076.97,2023
O00010,C00702,P0258,2023-07-07T00:00:00.000Z,4,5695.64,2023


In [0]:
df = df.withColumn("year", year(col("order_date")))

order_id,customer_id,product_id,order_date,quantity,total_amount,year
O00001,C00710,P0159,2023-03-22T00:00:00.000Z,3,2022.87,2023
O00002,C00954,P0036,2023-06-30T00:00:00.000Z,2,3560.74,2023
O00003,C01578,P0427,2023-11-06T00:00:00.000Z,3,5903.52,2023
O00004,C00962,P0332,2024-02-27T00:00:00.000Z,3,4107.99,2024
O00005,C00156,P0038,2024-10-13T00:00:00.000Z,5,5784.95,2024
O00006,C00521,P0174,2023-05-17T00:00:00.000Z,5,407.75,2023
O00007,C00982,P0352,2024-01-18T00:00:00.000Z,4,4907.64,2024
O00008,C00976,P0172,2023-01-10T00:00:00.000Z,4,7037.88,2023
O00009,C01001,P0238,2023-04-20T00:00:00.000Z,3,4076.97,2023
O00010,C00702,P0258,2023-07-07T00:00:00.000Z,4,5695.64,2023


In [0]:
class Window_func:
    def dense_rank(self,df):
        dense_ranked_df= df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return dense_ranked_df
    
    def rank(self,df):
        ranked_df = df.withColumn("rank_flag",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return ranked_df
    
    def row_number(self,df):
        row_numbered_df = df.withColumn("row_flag",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return row_numbered_df
    

In [0]:
df_new = df

In [0]:
obj = Window_func()
df_new = obj.dense_rank(df)
display(df_new.limit(10))

order_id,customer_id,product_id,order_date,quantity,total_amount,year,flag
O00957,C01449,P0498,2023-10-05T00:00:00.000Z,5,9952.9,2023,1
O01765,C01515,P0498,2023-05-11T00:00:00.000Z,5,9952.9,2023,1
O03502,C00805,P0498,2023-09-24T00:00:00.000Z,5,9952.9,2023,1
O03660,C01001,P0498,2023-10-08T00:00:00.000Z,5,9952.9,2023,1
O06790,C01819,P0498,2023-02-27T00:00:00.000Z,5,9952.9,2023,1
O03989,C00631,P0440,2023-11-20T00:00:00.000Z,5,9916.25,2023,2
O07763,C00471,P0440,2023-04-12T00:00:00.000Z,5,9916.25,2023,2
O03272,C01879,P0165,2023-02-15T00:00:00.000Z,5,9858.85,2023,3
O03746,C01713,P0165,2023-10-26T00:00:00.000Z,5,9858.85,2023,3
O08261,C00988,P0165,2023-11-29T00:00:00.000Z,5,9858.85,2023,3


###**Data writing**


In [0]:
df.write.format('delta').mode('append').save('abfss://silver@databrcks.dfs.core.windows.net/orders')